# Triaxial cluster shapes — 3D visualisation with lensing κ

Each galaxy cluster halo is modelled as a triaxial ellipsoid with semi-axes
$a \geq b \geq c$, where $a = r_{\rm vir}$ is the virial radius.
The axis ratios $q_{ba} = b/a$ and $q_{ca} = c/a$ are drawn from uniform
distributions, and the line-of-sight (LOS) direction is isotropic.

Each panel shows:
- **3D ellipsoid** (surface) in the frame where the LOS is the $+z$ axis
- **Lensing convergence κ** heatmap on the floor plane $z = z_{\rm floor}$,
  computed from the projected NFW profile with the correct elliptical symmetry
- **LOS arrow** (black, pointing downward toward the observer)
- **Principal axes** (coloured sticks)

All functions are defined inline to match the pipeline exactly.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401 – registers 3d projection
from scipy.spatial.transform import Rotation
from astropy.cosmology import Planck18

## Pipeline functions (defined inline)

All functions below are exact copies of the implementations in
`xlensing.cosmo`, `xlensing.sims`, `xlensing.model`, and `destacking.relations`.

In [2]:
# ── Cosmological constants (xlensing.cosmo) ───────────────────────────────────
COSMO               = Planck18
CLUSTER_OVERDENSITY = 200
OM0                 = COSMO.Om0
LIGHTSPEED          = 299792.458   # km s⁻¹
GRAVITY             = 4.302e-9     # km² s⁻² Mpc M☉⁻¹

def rhoM(z):
    """Matter density at z [M☉/Mpc³] — xlensing.cosmo.rhoM."""
    return OM0 * COSMO.critical_density(z).to('Msun/Mpc3').value

def r_vir(z, M):
    """Virial radius for mass M [M☉] at redshift z [Mpc] — xlensing.sims.r_vir."""
    return ((3.0 * M) / (rhoM(z) * 4.0 * np.pi * CLUSTER_OVERDENSITY)) ** (1.0 / 3.0)

def NFW_delta_c(conc):
    """NFW characteristic density ratio — xlensing.sims.NFW_delta_c."""
    return (CLUSTER_OVERDENSITY / 3.0) * conc**3 / (np.log(1.0 + conc) - conc / (1.0 + conc))

def c_DuttonMaccio(z, m, c=1., h=1.):
    """NFW concentration c200 from Dutton & Maccio (2014) — destacking.relations."""
    a = 0.52 + 0.385 * np.exp(-0.617 * (z ** 1.21))
    b = -0.101 + 0.026 * z
    return 10.0 ** (a + b * np.log10(m * h / (c * 1e12)))

def SigX(x):
    """Dimensionless NFW projected surface density kernel — xlensing.model.SigX."""
    x       = np.asarray(x, dtype=float)
    result  = np.zeros_like(x)
    lesser  = x < 1.0
    equal   = x == 1.0
    greater = x > 1.0
    result[equal]   = 1.0 / 3.0
    result[lesser]  = (1.0 - 2.0 / np.sqrt(1.0 - x[lesser]**2)
                       * np.arctanh(np.sqrt((1.0 - x[lesser]) / (1.0 + x[lesser])))) / (x[lesser]**2 - 1.0)
    result[greater] = (1.0 - 2.0 / np.sqrt(x[greater]**2 - 1.0)
                       * np.arctan(np.sqrt((x[greater] - 1.0) / (1.0 + x[greater])))) / (x[greater]**2 - 1.0)
    return result

def nfw_sigma(R, M200, c200, z):
    """NFW projected surface density Σ(R) [M☉/Mpc²] — derived from xlensing.model."""
    rs   = r_vir(z, M200) / c200
    fact = 2.0 * rs * NFW_delta_c(c200) * rhoM(z)
    x    = np.maximum(np.asarray(R, dtype=float), 1e-6) / rs
    return fact * SigX(x)

def sigma_crit(z_l, z_s):
    """Critical surface density [M☉/Mpc²] — xlensing.sims.critical_density."""
    DA  = lambda z1, z2: COSMO.angular_diameter_distance_z1z2(z1, z2).value
    Ds  = DA(0.0, z_s)
    Dl  = DA(0.0, z_l)
    Dls = DA(z_l, z_s)
    return (LIGHTSPEED**2 / (4.0 * np.pi * GRAVITY)) * Ds / (Dl * Dls)

def project_ellipsoid(a, b, c, theta_los, phi_los):
    """Project triaxial ellipsoid onto sky plane (Schur complement) — xlensing.sims."""
    sth, cth = np.sin(theta_los), np.cos(theta_los)
    sph, cph = np.sin(phi_los),   np.cos(phi_los)
    n        = np.array([sth * cph, sth * sph, cth])
    Q        = np.diag([1.0 / a**2, 1.0 / b**2, 1.0 / c**2])
    e1 = np.array([1.0, 0.0, 0.0]) - float(np.dot([1.0, 0.0, 0.0], n)) * n
    if np.linalg.norm(e1) < 1e-10:
        e1 = np.array([0.0, 1.0, 0.0]) - float(np.dot([0.0, 1.0, 0.0], n)) * n
    e1 /= np.linalg.norm(e1)
    e2  = np.cross(n, e1)
    Qn  = Q @ n
    nQn = float(n @ Qn)
    M_mat = np.array([
        [e1 @ Q @ e1 - (e1 @ Qn)**2 / nQn,
         e1 @ Q @ e2 - (e1 @ Qn) * (e2 @ Qn) / nQn],
        [e2 @ Q @ e1 - (e2 @ Qn) * (e1 @ Qn) / nQn,
         e2 @ Q @ e2 - (e2 @ Qn)**2 / nQn],
    ])
    eigvals, eigvecs = np.linalg.eigh(M_mat)
    q_maj = 1.0 / np.sqrt(eigvals[0])
    q_min = 1.0 / np.sqrt(eigvals[1])
    pa    = np.arctan2(eigvecs[1, 0], eigvecs[0, 0])
    return q_maj, q_min, pa

def sample_triaxial_shapes(z, M, ba_range=(0.5, 1.0), ca_lo=0.3,
                            isotropic_los=True, rng=None):
    """Sample CDM-like triaxial halo shapes and project — xlensing.sims."""
    z = np.atleast_1d(np.asarray(z, dtype=float))
    M = np.atleast_1d(np.asarray(M, dtype=float))
    N = len(z)
    if rng is None:
        rng = np.random.default_rng()
    r_vir_vals = np.array([r_vir(float(zi), float(Mi)) for zi, Mi in zip(z, M)])
    q_ba = rng.uniform(ba_range[0], ba_range[1], N)
    q_ca = rng.uniform(ca_lo, q_ba)
    if isotropic_los:
        theta_los = np.arccos(rng.uniform(0.0, 1.0, N))
    else:
        theta_los = np.zeros(N)
    phi_los = rng.uniform(0.0, 2.0 * np.pi, N)
    q_maj = np.zeros(N)
    q_min = np.zeros(N)
    pa    = np.zeros(N)
    for i in range(N):
        a = r_vir_vals[i]
        b = q_ba[i] * a
        c = q_ca[i] * a
        q_maj[i], q_min[i], pa[i] = project_ellipsoid(a, b, c, theta_los[i], phi_los[i])
    return {
        'q_ba':      q_ba,
        'q_ca':      q_ca,
        'theta_los': theta_los,
        'phi_los':   phi_los,
        'r_vir':     r_vir_vals,
        'q_maj':     q_maj,
        'q_min':     q_min,
        'pa':        pa,
    }

def kappa_grid(M200, c200, z_l, z_s, q_maj, q_min, pa, n_grid=60, r_max=None):
    """Lensing convergence κ on a 2D sky-plane grid for an elliptically projected NFW.

    Convention matches triaxial_NFW_wcs_cluster_shear: u along major axis,
    v along minor axis, with r_eff = sqrt((u/q_maj)²+(v/q_min)²)*sqrt(q_maj*q_min).
    Returns (xv, yv, kappa) in Mpc / dimensionless.
    """
    if r_max is None:
        r_max = r_vir(z_l, M200) * 1.35
    xv     = np.linspace(-r_max, r_max, n_grid)
    yv     = np.linspace(-r_max, r_max, n_grid)
    Xv, Yv = np.meshgrid(xv, yv)
    u      = Xv * np.sin(pa) + Yv * np.cos(pa)
    v      = Xv * np.cos(pa) - Yv * np.sin(pa)
    r_eff  = np.sqrt((u / q_maj)**2 + (v / q_min)**2) * np.sqrt(q_maj * q_min)
    r_eff  = np.maximum(r_eff, 1e-4)
    sigma  = nfw_sigma(r_eff, M200, c200, z_l)
    sc     = sigma_crit(z_l, z_s)
    return xv, yv, sigma / sc

print('Pipeline functions defined.')

Pipeline functions defined.


## 1  Generate a sample of 16 triaxial shapes

We draw clusters uniformly in $z \in [0.2, 0.65]$ and
$\log_{10} M \in [13.5, 15.0]\,M_\odot$,
matching the DES Y3 range used in the pipeline.

In [3]:
N   = 16
rng = np.random.default_rng(2026)

z_sample = rng.uniform(0.20, 0.65, N)
M_sample = 10 ** rng.uniform(13.5, 15.0, N)

shapes = sample_triaxial_shapes(
    z_sample, M_sample,
    ba_range=(0.5, 1.0),
    ca_lo=0.3,
    isotropic_los=True,
    rng=rng,
)

print(f"{'#':>3}  {'z':>5}  {'logM':>6}  {'r_vir':>7}  "
      f"{'b/a':>5}  {'c/a':>5}  {'theta_los':>9}  {'q_maj':>7}  {'q_min':>7}")
print("-" * 70)
for i in range(N):
    print(
        f"{i:>3d}  {z_sample[i]:>5.3f}  {np.log10(M_sample[i]):>6.2f}  "
        f"{shapes['r_vir'][i]:>7.3f}  "
        f"{shapes['q_ba'][i]:>5.3f}  {shapes['q_ca'][i]:>5.3f}  "
        f"{np.degrees(shapes['theta_los'][i]):>8.1f}°  "
        f"{shapes['q_maj'][i]:>7.3f}  {shapes['q_min'][i]:>7.3f}"
    )

  #      z    logM    r_vir    b/a    c/a  theta_los    q_maj    q_min
----------------------------------------------------------------------
  0  0.281   14.17    1.498  0.899  0.738      25.2°    1.496    1.307
  1  0.488   14.01    1.217  0.803  0.576       8.2°    1.215    0.974
  2  0.410   13.92    1.171  0.673  0.532      86.6°    0.790    0.625
  3  0.367   13.84    1.122  0.973  0.551      69.0°    1.102    0.701
  4  0.360   14.29    1.589  0.782  0.502      43.1°    1.291    1.233
  5  0.556   14.15    1.317  0.716  0.506      71.7°    1.108    0.719
  6  0.607   14.49    1.685  0.950  0.606      55.5°    1.601    1.272
  7  0.280   13.52    0.908  0.660  0.543      65.4°    0.747    0.529
  8  0.494   14.17    1.377  0.848  0.616      39.3°    1.250    1.121
  9  0.334   14.05    1.334  0.657  0.449      16.6°    1.308    0.868
 10  0.635   13.79    0.972  0.631  0.301      27.3°    0.901    0.597
 11  0.614   14.39    1.553  0.850  0.737      51.6°    1.531    1.221
 12  0

## 2  Rotation and surface helpers

In [4]:
def rotation_to_los_frame(theta_los, phi_los):
    """Return 3×3 rotation matrix R such that R @ n_los = [0, 0, 1]."""
    n = np.array([
        np.sin(theta_los) * np.cos(phi_los),
        np.sin(theta_los) * np.sin(phi_los),
        np.cos(theta_los),
    ])
    z_hat = np.array([0., 0., 1.])
    cross = np.cross(n, z_hat)
    sin_a = np.linalg.norm(cross)
    cos_a = float(np.dot(n, z_hat))
    if sin_a < 1e-10:
        return np.eye(3) if cos_a > 0 else np.diag([1., -1., -1.])
    axis  = cross / sin_a
    angle = np.arctan2(sin_a, cos_a)
    return Rotation.from_rotvec(angle * axis).as_matrix()


def ellipsoid_surface(a, b, c, nu=50, nv=40):
    """Parametric ellipsoid surface in the principal-axes frame."""
    u = np.linspace(0, 2 * np.pi, nu)
    v = np.linspace(0, np.pi,     nv)
    x = a * np.outer(np.cos(u), np.sin(v))
    y = b * np.outer(np.sin(u), np.sin(v))
    z = c * np.outer(np.ones(nu), np.cos(v))
    return x, y, z


def rotate_surface(x, y, z, R):
    """Apply rotation matrix R to a (nu, nv) surface array."""
    pts   = np.stack([x, y, z], axis=-1)
    pts_r = (R @ pts.reshape(-1, 3).T).T
    sh    = x.shape
    return pts_r[:, 0].reshape(sh), pts_r[:, 1].reshape(sh), pts_r[:, 2].reshape(sh)


def sky_ellipse(q_maj, q_min, pa, n_pts=300):
    """Sky-plane ellipse coordinates (used for LOS-angle plot)."""
    t  = np.linspace(0, 2 * np.pi, n_pts)
    xp = q_maj * np.cos(t) * np.cos(pa) - q_min * np.sin(t) * np.sin(pa)
    yp = q_maj * np.cos(t) * np.sin(pa) + q_min * np.sin(t) * np.cos(pa)
    return xp, yp

## 3  4×4 panel: 3D ellipsoids with lensing κ

In [ ]:
Z_SRC         = 1.0    # background source redshift for κ
LOG_KAPPA_MIN = -3.0
LOG_KAPPA_MAX =  0.0
N_KAPPA_GRID  = 48
KAPPA_LEVELS  = np.linspace(LOG_KAPPA_MIN, LOG_KAPPA_MAX, 60)

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

norm_kappa = Normalize(vmin=LOG_KAPPA_MIN, vmax=LOG_KAPPA_MAX)
cmap_kappa = plt.cm.inferno
cmap_shape = plt.cm.viridis

fig = plt.figure(figsize=(22, 22))
fig.patch.set_facecolor('white')

for i in range(N):
    a         = shapes['r_vir'][i]
    b         = shapes['q_ba'][i] * a
    c_ax      = shapes['q_ca'][i] * a
    theta_los = shapes['theta_los'][i]
    phi_los   = shapes['phi_los'][i]
    q_maj     = shapes['q_maj'][i]
    q_min     = shapes['q_min'][i]
    pa        = shapes['pa'][i]
    c200_i    = c_DuttonMaccio(z_sample[i], M_sample[i])

    R_rot = rotation_to_los_frame(theta_los, phi_los)

    xe, ye, ze = ellipsoid_surface(a, b, c_ax)
    xr, yr, zr = rotate_surface(xe, ye, ze, R_rot)

    z_floor = -a * 1.55

    xv, yv, kappa = kappa_grid(M_sample[i], c200_i, z_sample[i], Z_SRC,
                                q_maj, q_min, pa, n_grid=N_KAPPA_GRID, r_max=a * 1.35)
    log_kappa = np.log10(np.clip(kappa, 10**LOG_KAPPA_MIN, 10**LOG_KAPPA_MAX))
    Xv, Yv    = np.meshgrid(xv, yv)

    ax = fig.add_subplot(4, 4, i + 1, projection='3d')
    ax.set_facecolor('white')

    ax.plot_surface(
        xr, yr, zr,
        color=cmap_shape(shapes['q_ba'][i] * shapes['q_ca'][i]),
        alpha=0.55, linewidth=0, antialiased=True, rstride=1, cstride=1,
    )

    ax.contourf(
        Xv, Yv, log_kappa,
        levels=KAPPA_LEVELS, cmap=cmap_kappa, norm=norm_kappa,
        zdir='z', offset=z_floor, alpha=0.92,
    )

    # drop-lines from ellipsoid equator to floor
    equator_u = np.linspace(0, 2 * np.pi, 24, endpoint=False)
    eq_x = a * np.cos(equator_u)
    eq_y = b * np.sin(equator_u)
    eq_z = np.zeros_like(equator_u)
    eq_pts = rotate_surface(eq_x[np.newaxis, :], eq_y[np.newaxis, :], eq_z[np.newaxis, :], R_rot)
    for j in range(0, 24, 4):
        ax.plot(
            [eq_pts[0][0, j], eq_pts[0][0, j]],
            [eq_pts[1][0, j], eq_pts[1][0, j]],
            [eq_pts[2][0, j], z_floor],
            color='gray', alpha=0.25, lw=0.7, ls='--',
        )

    # principal axes
    axes_tips = R_rot @ np.array([[a, 0, 0], [0, b, 0], [0, 0, c_ax]]).T
    for k, col in enumerate(['#cc2222', '#228822', '#2255cc']):
        ax.quiver(
            0, 0, 0, axes_tips[0, k], axes_tips[1, k], axes_tips[2, k],
            color=col, lw=2.0, arrow_length_ratio=0.2, alpha=0.9,
        )

    # LOS arrow
    los_top = a * 1.3
    ax.quiver(0, 0, los_top, 0, 0, -(los_top - z_floor) * 0.45,
              color='black', lw=2.5, arrow_length_ratio=0.12, alpha=0.85)
    ax.text(0.1 * a, 0, los_top * 1.05, 'LOS', color='black',
            fontsize=6.5, ha='left', va='bottom')

    eps = 1.0 - q_min / q_maj
    ax.text(0, 0, z_floor - a * 0.05, rf'$\varepsilon={eps:.2f}$',
            color='#882222', fontsize=6.5, ha='center', va='top')

    lim = a * 1.4
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(z_floor - a * 0.1, los_top * 1.1)
    ax.set_xlabel('x', fontsize=6.5, color='black', labelpad=2)
    ax.set_ylabel('y', fontsize=6.5, color='black', labelpad=2)
    ax.set_zlabel('z (LOS)', fontsize=6.5, color='black', labelpad=2)
    ax.tick_params(labelsize=5.5, colors='black', pad=1)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('#cccccc')
    ax.yaxis.pane.set_edgecolor('#cccccc')
    ax.zaxis.pane.set_edgecolor('#cccccc')
    ax.grid(False)
    ax.view_init(elev=22, azim=35)
    ax.set_title(
        f'#{i+1}   z={z_sample[i]:.2f}   '
        f'$\\log M={np.log10(M_sample[i]):.1f}$\n'
        f'$b/a={shapes["q_ba"][i]:.2f}$   '
        f'$c/a={shapes["q_ca"][i]:.2f}$   '
        f'$\\theta_{{\\rm los}}={np.degrees(theta_los):.0f}°$',
        fontsize=7.5, color='black', pad=4,
    )

# shared κ colorbar
# sm = ScalarMappable(cmap=cmap_kappa, norm=norm_kappa)
# sm.set_array([])
# cbar_ax = fig.add_axes([0.92, 0.10, 0.015, 0.75])
# cbar = fig.colorbar(sm, cax=cbar_ax)
# cbar.set_label(r'$\log_{10}\,\kappa$', fontsize=11)
# cbar.ax.tick_params(labelsize=9)

legend_elements = [
    Line2D([0], [0], color='#cc2222', lw=2, label='Semi-axis $a$ (major)'),
    Line2D([0], [0], color='#228822', lw=2, label='Semi-axis $b$ (intermediate)'),
    Line2D([0], [0], color='#2255cc', lw=2, label='Semi-axis $c$ (minor)'),
    Line2D([0], [0], color='black',   lw=2, label='Line of sight'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           fontsize=10, facecolor='white', edgecolor='#aaaaaa',
           bbox_to_anchor=(0.5, -0.01))

fig.suptitle(
    'Triaxial cluster shapes — 3D ellipsoids with lensing convergence κ\n'
    r'$(b/a \sim U[0.5,1.0],\; c/a \sim U[0.3,b/a],\; '
    r'\theta_{\rm los}\; {\rm isotropic})$',
    fontsize=14, color='black', y=1.01,
)
plt.tight_layout()
plt.savefig('figs/triaxial_shapes_3d.png', dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Saved figs/triaxial_shapes_3d.png')

## 4  Axis-ratio distributions

In [ ]:
N_large = 5000
rng2    = np.random.default_rng(999)
z_large = rng2.uniform(0.2, 0.65, N_large)
M_large = 10 ** rng2.uniform(13.5, 15.0, N_large)
sh_large = sample_triaxial_shapes(
    z_large, M_large, ba_range=(0.5, 1.0), ca_lo=0.3,
    isotropic_los=True, rng=rng2,
)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.patch.set_facecolor('white')
for ax in axes:
    ax.set_facecolor('white')

axes[0].hist(sh_large['q_ba'], bins=30, color='#4488cc', edgecolor='white', density=True)
axes[0].set(xlabel='$b/a$', ylabel='PDF', title='Intermediate/major ratio')

axes[1].hist(sh_large['q_ca'], bins=30, color='#44aa66', edgecolor='white', density=True)
axes[1].set(xlabel='$c/a$', title='Minor/major ratio')

axes[2].hist(np.degrees(sh_large['theta_los']), bins=30,
             color='#cc8844', edgecolor='white', density=True)
axes[2].set(xlabel=r'$\theta_{\rm los}$ [deg]', title='LOS inclination')
th = np.linspace(0, 90, 200)
axes[2].plot(th, np.sin(np.radians(th)) * np.pi / 180, 'k--', lw=1.5, label='sin θ')
axes[2].legend(fontsize=9)

eps = 1.0 - sh_large['q_min'] / sh_large['q_maj']
axes[3].hist(eps, bins=30, color='#cc4444', edgecolor='white', density=True)
axes[3].set(xlabel=r'$\varepsilon = 1 - q_{\rm min}/q_{\rm maj}$',
            title='Sky-plane ellipticity')

for ax in axes:
    ax.set_ylabel('PDF')

plt.suptitle(f'Shape distributions — {N_large} clusters', y=1.02)
plt.tight_layout()
plt.savefig('figs/triaxial_shape_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 5  Effect of LOS angle on the projected profile

Hold one cluster fixed and vary $\theta_{\rm los}$ from face-on ($0°$, LOS along $c$-axis)
to edge-on ($90°$, LOS perpendicular to $c$).

In [ ]:
a_fix, q_ba_fix, q_ca_fix = 1.0, 0.65, 0.40
b_fix = q_ba_fix * a_fix
c_fix = q_ca_fix * a_fix
z_fix, M_fix = 0.35, 3e14
c200_fix = c_DuttonMaccio(z_fix, M_fix)

theta_vals = np.radians(np.arange(0, 91, 10))
phi_fix    = 0.0

n_angles = len(theta_vals)
fig2 = plt.figure(figsize=(20, 4.5))
fig2.patch.set_facecolor('white')

KAPPA_LEVELS_LOS = np.linspace(LOG_KAPPA_MIN, LOG_KAPPA_MAX, 60)

for j, th in enumerate(theta_vals):
    qmaj, qmin, pa_j = project_ellipsoid(a_fix, b_fix, c_fix, th, phi_fix)
    eps_j = 1.0 - qmin / qmaj

    R_rot = rotation_to_los_frame(th, phi_fix)
    xe, ye, ze = ellipsoid_surface(a_fix, b_fix, c_fix, nu=40, nv=30)
    xr, yr, zr = rotate_surface(xe, ye, ze, R_rot)

    z_floor = -a_fix * 1.5

    xv, yv, kappa_j = kappa_grid(M_fix, c200_fix, z_fix, Z_SRC,
                                  qmaj, qmin, pa_j, n_grid=40, r_max=a_fix * 1.35)
    log_kappa_j = np.log10(np.clip(kappa_j, 10**LOG_KAPPA_MIN, 10**LOG_KAPPA_MAX))
    Xv2, Yv2 = np.meshgrid(xv, yv)

    ax = fig2.add_subplot(1, n_angles, j + 1, projection='3d')
    ax.set_facecolor('white')

    ax.plot_surface(xr, yr, zr, color='#5599ff', alpha=0.55, linewidth=0)
    ax.contourf(Xv2, Yv2, log_kappa_j,
                levels=KAPPA_LEVELS_LOS, cmap=cmap_kappa, norm=norm_kappa,
                zdir='z', offset=z_floor, alpha=0.92)

    lim = a_fix * 1.4
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(z_floor - 0.1, a_fix * 1.3)
    ax.set_axis_off()
    ax.view_init(elev=22, azim=35)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    ax.set_title(
        f'$\\theta_{{\\rm los}}={np.degrees(th):.0f}°$\n'
        f'$\\varepsilon={eps_j:.2f}$',
        fontsize=8.5, color='black', pad=3,
    )

fig2.suptitle(
    f'Projection dependence on LOS angle  '
    f'($b/a={q_ba_fix}$, $c/a={q_ca_fix}$)',
    fontsize=12, color='black', y=1.02,
)
plt.tight_layout()
plt.savefig('figs/triaxial_los_angle_effect.png', dpi=130,
            bbox_inches='tight', facecolor=fig2.get_facecolor())
plt.show()
print('Saved figs/triaxial_los_angle_effect.png')